# Payload 3D Orientation Animator

This notebook creates 3D animations showing the payload orientation (yaw/pitch/roll) over time by animating a simple rocket shape.

## Setup and Library Installation

Install required libraries for 3D visualization and animation.

In [92]:
# Install required packages
import subprocess
import sys

packages = ['matplotlib', 'pandas', 'numpy']
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("All packages installed successfully")

All packages installed successfully


## Import Libraries

Import necessary libraries for data processing and 3D animation.

In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path
from io import StringIO

print("Libraries imported successfully")

Libraries imported successfully


## Load Flight Data

Load and parse flight log data, splitting into separate flights.

In [94]:
def load_flight_data(filepath):
    """
    Load flight data from a txt file and split into separate flights.
    Returns a list of DataFrames, one for each flight.
    """
    flights = []
    current_flight = []
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Skip header and parse data
    header_line = None
    for i, line in enumerate(lines):
        line = line.strip()
        
        # Store header
        if 'millis' in line and header_line is None:
            header_line = line
            continue
        
        # Check for end of flight marker
        if '[---END OF FLIGHT---]' in line:
            if current_flight:
                data_str = header_line + '\n' + '\n'.join(current_flight)
                df = pd.read_csv(StringIO(data_str))
                # Convert all numeric columns to proper types
                df['millis'] = pd.to_numeric(df['millis'], errors='coerce')
                df['yaw_deg'] = pd.to_numeric(df['yaw_deg'], errors='coerce')
                df['pitch_deg'] = pd.to_numeric(df['pitch_deg'], errors='coerce')
                df['roll_deg'] = pd.to_numeric(df['roll_deg'], errors='coerce')
                df['altitude_ft'] = pd.to_numeric(df['altitude_ft'], errors='coerce')
                flights.append(df)
                current_flight = []
            continue
        
        # Add data line to current flight
        if line and not line.startswith('['):
            current_flight.append(line)
    
    # Handle last flight if no end marker
    if current_flight:
        data_str = header_line + '\n' + '\n'.join(current_flight)
        df = pd.read_csv(StringIO(data_str))
        # Convert all numeric columns to proper types
        df['millis'] = pd.to_numeric(df['millis'], errors='coerce')
        df['yaw_deg'] = pd.to_numeric(df['yaw_deg'], errors='coerce')
        df['pitch_deg'] = pd.to_numeric(df['pitch_deg'], errors='coerce')
        df['roll_deg'] = pd.to_numeric(df['roll_deg'], errors='coerce')
        df['altitude_ft'] = pd.to_numeric(df['altitude_ft'], errors='coerce')
        flights.append(df)
    
    return flights

# Load the flight data
log_file = "flight_log_11_22_25_pt1.txt"
# Ensure you have the 'data' folder or adjust path as needed
flights = load_flight_data(f"data/{log_file}")
filename = Path(log_file).stem

# Create output directory
output_dir = Path("animations") / filename
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Loaded {len(flights)} flight(s) from {log_file}")
for i, flight in enumerate(flights, 1):
    print(f"  Flight {i}: {len(flight)} data points")

/var/folders/91/k7skb57154qb4889c46jt8th0000gn/T/ipykernel_86630/2707451465.py:26: DtypeWarning: Columns (3,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(StringIO(data_str))


Loaded 4 flight(s) from flight_log_11_22_25_pt1.txt
  Flight 1: 46863 data points
  Flight 2: 568246 data points
  Flight 3: 375376 data points
  Flight 4: 1025 data points


## Define Rotation Functions

Functions to apply yaw, pitch, and roll rotations to 3D coordinates.

In [95]:
def rotation_matrix_y(angle):
    """Rotation matrix for pitch (rotation around Y-axis)."""
    angle = np.radians(angle)
    return np.array([
        [np.cos(angle), 0, np.sin(angle)],
        [0, 1, 0],
        [-np.sin(angle), 0, np.cos(angle)]
    ])

def rotation_matrix_z(angle):
    """Rotation matrix for yaw/roll (rotation around Z-axis)."""
    angle = np.radians(angle)
    return np.array([
        [np.cos(angle), -np.sin(angle), 0],
        [np.sin(angle), np.cos(angle), 0],
        [0, 0, 1]
    ])

def apply_custom_rotation(vertices, yaw_sensor, pitch_sensor, roll_sensor):
    """
    Apply rotations for a Z-aligned rocket where -90 pitch is vertical.
    
    Kinematic Chain:
    1. ROLL (Spin): Rotate around Z (Body Axis) by `roll_sensor`.
    2. PITCH (Tilt): Rotate around Y by `pitch_sensor + 90`. 
       (Since -90 is vertical, -90+90=0, so no tilt happens -> stays vertical).
    3. YAW (Heading): Rotate around Z (World Axis) by `yaw_sensor`.
    """
    # 1. Body Spin (Sensor Roll -> Z Rotation)
    R_spin = rotation_matrix_z(roll_sensor)
    
    # 2. Tilt (Sensor Pitch -> Y Rotation with offset)
    # If pitch is -90, we want 0 rotation (Vertical).
    # If pitch is 0, we want 90 rotation (Horizontal).
    R_tilt = rotation_matrix_y(pitch_sensor + 90)
    
    # 3. Heading (Sensor Yaw -> Z Rotation)
    R_heading = rotation_matrix_z(yaw_sensor)
    
    # Combine: Heading * Tilt * Spin
    R_total = R_heading @ R_tilt @ R_spin
    
    # Apply rotation to all vertices
    return np.dot(vertices, R_total.T)

print("Rotation functions defined")

Rotation functions defined


## Create Rocket Geometry (Z-Aligned)

Define the vertices and faces of a simple rocket shape.  
**FIX:** Rocket is now built along the Z-axis (Vertical).

In [96]:
def create_rocket_vertices(length=2, radius=0.3):
    """
    Create vertices for a simple rocket shape centered at origin.
    **ROCKET IS ALIGNED ALONG Z-AXIS (Vertical)**
    """
    # Body cylinder vertices (8 points around circle at 2 heights)
    angles = np.linspace(0, 2*np.pi, 9)[:-1]  # 8 points around circle
    
    body_bottom_z = -length/2
    body_top_z = length/3
    nose_tip_z = length/2
    
    # Body bottom circle
    body_bottom = np.array([[radius * np.cos(a), radius * np.sin(a), body_bottom_z] for a in angles])
    
    # Body top circle (where cone starts)
    body_top = np.array([[radius * np.cos(a), radius * np.sin(a), body_top_z] for a in angles])
    
    # Nose cone tip
    nose_tip = np.array([[0, 0, nose_tip_z]])
    
    # Combine all vertices
    vertices = np.vstack([body_bottom, body_top, nose_tip])
    
    return vertices

def get_rocket_faces(vertices):
    """Define the faces of the rocket."""
    faces = []
    
    # Body cylinder sides (8 rectangular faces)
    for i in range(8):
        next_i = (i + 1) % 8
        face = [
            vertices[i],           # bottom current
            vertices[next_i],      # bottom next
            vertices[8 + next_i],  # top next
            vertices[8 + i]        # top current
        ]
        faces.append(face)
    
    # Body bottom face (octagon)
    bottom_face = [vertices[i] for i in range(8)]
    faces.append(bottom_face)
    
    # Nose cone (8 triangular faces)
    nose_tip_idx = 16
    for i in range(8):
        next_i = (i + 1) % 8
        face = [
            vertices[8 + i],       # body top current
            vertices[8 + next_i],  # body top next
            vertices[nose_tip_idx] # nose tip
        ]
        faces.append(face)
    
    return faces

print("Rocket geometry functions defined (Z-axis aligned)")

Rocket geometry functions defined (Z-axis aligned)


## Create Animation

Generate 3D animation showing payload orientation over time.
**Update:** Fixed kinematic chain for vertical Z alignment.

In [97]:
def create_orientation_animation(df, flight_num, filename, skip_frames=10, start_time_seconds=None):
    """
    Create 3D animation of payload orientation.
    """
    # Filter data by start time if specified
    if start_time_seconds is not None:
        start_millis = start_time_seconds * 1000
        df = df[df['millis'] >= start_millis].reset_index(drop=True)
        print(f"Filtered to data starting at {start_time_seconds}s ({len(df)} points)")
    
    # Downsample data for animation
    df_sampled = df.iloc[::skip_frames].reset_index(drop=True)
    
    print(f"Creating animation for Flight {flight_num} ({len(df_sampled)} frames)...")
    
    # Create figure and 3D axis
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Set up the plot limits and labels
    ax.set_xlim([-2, 2])
    ax.set_ylim([-2, 2])
    ax.set_zlim([-2, 2])
    
    # --- UPDATED AXES LABELS ---
    ax.set_xlabel('X (Ground/East)', fontsize=10)
    ax.set_ylabel('Y (Ground/North)', fontsize=10)
    ax.set_zlabel('Z (Altitude/Vertical)', fontsize=10)
    
    # Create initial rocket ALIGNED TO Z-AXIS
    base_vertices = create_rocket_vertices()
    
    # Initialize collection for the rocket
    poly_collection = Poly3DCollection([], alpha=0.7, edgecolors='black', linewidths=1)
    ax.add_collection3d(poly_collection)
    
    # Text annotations
    time_text = ax.text2D(0.02, 0.95, '', transform=ax.transAxes, fontsize=10)
    yaw_text = ax.text2D(0.02, 0.90, '', transform=ax.transAxes, fontsize=9)
    pitch_text = ax.text2D(0.02, 0.85, '', transform=ax.transAxes, fontsize=9)
    roll_text = ax.text2D(0.02, 0.80, '', transform=ax.transAxes, fontsize=9)
    altitude_text = ax.text2D(0.02, 0.75, '', transform=ax.transAxes, fontsize=9)
    state_text = ax.text2D(0.02, 0.70, '', transform=ax.transAxes, fontsize=9, fontweight='bold')
    
    # Add legend text
    ax.text2D(0.98, 0.95, 'Red Arrow = Nose Direction', transform=ax.transAxes, fontsize=8, ha='right', color='red')
    
    arrows = []
    
    def update(frame):
        nonlocal arrows
        
        # Remove previous arrows
        for arrow in arrows:
            arrow.remove()
        arrows.clear()
        
        row = df_sampled.iloc[frame]
        
        yaw = row['yaw_deg']
        pitch = row['pitch_deg']
        roll = row['roll_deg']
        
        # Apply custom rotation chain
        rotated_vertices = apply_custom_rotation(base_vertices, yaw, pitch, roll)
        
        # Get faces for the rotated rocket
        faces = get_rocket_faces(rotated_vertices)
        
        # Update polygon collection
        poly_collection.set_verts(faces)
        
        # Color based on state
        state_colors = {
            'GROUND': '#808080',
            'ASCENT': '#3498db',
            'DESCENT': '#e74c3c',
            'LANDED': '#95a5a6'
        }
        color = state_colors.get(row['state'], '#2ecc71')
        poly_collection.set_facecolor(color)
        
        # Create "UP" arrow pointing along rocket's long axis from nose tip
        nose_tip = rotated_vertices[16]  # Index 16 is the nose tip

        # The Nose direction in the BASE Z-aligned model is [0, 0, 1]
        nose_direction_base = np.array([[0, 0, 1.5]])
        
        # Apply same rotation to arrow vector
        nose_vector_rotated = apply_custom_rotation(nose_direction_base, yaw, pitch, roll)[0]
        
        up_arrow = ax.quiver(
            nose_tip[0], nose_tip[1], nose_tip[2],
            nose_vector_rotated[0], nose_vector_rotated[1], nose_vector_rotated[2],
            color='red', arrow_length_ratio=0.2, linewidth=2.0
        )
        arrows.append(up_arrow)
        
        # Update text
        time_seconds = row['millis'] / 1000
        time_text.set_text(f'Time: {time_seconds:.1f}s')
        yaw_text.set_text(f'Sensor Yaw (Z): {row["yaw_deg"]:.1f}°')
        pitch_text.set_text(f'Sensor Pitch (Y): {row["pitch_deg"]:.1f}°')
        roll_text.set_text(f'Sensor Roll (X): {row["roll_deg"]:.1f}°')
        altitude_text.set_text(f'Altitude: {row["altitude_ft"]:.1f} ft')
        state_text.set_text(f'State: {row["state"]}')
        state_text.set_color(color)
        
        return [poly_collection] + arrows + [time_text, yaw_text, pitch_text, roll_text, altitude_text, state_text]
    
    # Create animation
    anim = FuncAnimation(
        fig, 
        update, 
        frames=len(df_sampled),
        interval=50,  # 50ms between frames = 20 fps
        blit=False
    )
    
    # Save animation
    output_path = output_dir / f"{filename}_orientation_f{flight_num}.gif"
    writer = PillowWriter(fps=20)
    anim.save(output_path, writer=writer)
    
    plt.close()
    print(f"Saved: {output_path}")

## Generate Animations for All Flights

Create orientation animations for each flight in the log file.

In [98]:
# Generate animations (using skip_frames=20 to reduce file size)
# Set start_time_seconds to filter data (e.g., 60 to start at 60 seconds)

start_time = 4545  # Change this value or set to None to start from beginning

# to output single animation.
create_orientation_animation(flights[2], 3, filename, skip_frames=10, 
                             start_time_seconds=start_time)

print(f"\nAll animations saved to: {output_dir}")

Filtered to data starting at 4545s (7705 points)
Creating animation for Flight 3 (771 frames)...
Saved: animations/flight_log_11_22_25_pt1/flight_log_11_22_25_pt1_orientation_f3.gif

All animations saved to: animations/flight_log_11_22_25_pt1


## Summary

3D orientation animations have been generated and saved to the `animations/` directory. Each animation shows:
- A simple 3D rocket shape representing the payload
- Real-time orientation based on yaw, pitch, and roll data
- Color-coded by flight state (GROUND, ASCENT, DESCENT, LANDED)
- Red arrow showing "UP" direction from nose tip
- Blue arrow showing "FORWARD" direction
- Current time and orientation angles displayed on screen

**Note:** Animations are saved as GIF files. For smoother playback, consider reducing `skip_frames` parameter (will increase file size).